In [ ]:
import kaggle_benchmarks as kbench
import json
import re
from datetime import datetime

# ----------------------------
# Global trace store
# ----------------------------
TRACE_LOG = []

FAILURE_MODES = [
    "failure_to_recognize_key_aspects",
    "hallucination",
    "misapplication_of_equation_or_model",
    "incorrect_factual_knowledge",
    "calculation_error",
]

CANONICAL_NUMERIC_ANSWER = 0.418
ABS_TOL = 0.005  # enough for 3 s.f. acceptance

# ----------------------------
# Helpers
# ----------------------------
def extract_json(text):
    if not text:
        return None

    fence = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if fence:
        blob = fence.group(1)
    else:
        start = text.find("{")
        end = text.rfind("}")
        if start == -1 or end == -1 or end <= start:
            return None
        blob = text[start:end + 1]

    try:
        return json.loads(blob)
    except Exception:
        return None

def safe_get_attr(obj, attr_name, default=None):
    try:
        return getattr(obj, attr_name, default)
    except Exception:
        return default

def build_trace(
    *,
    task_id,
    llm,
    prompt,
    response,
    parsed,
    final_answer,
    normalized_answer,
    passed,
    failure_mode,
):
    return {
        "timestamp_utc": datetime.utcnow().isoformat() + "Z",
        "task_id": task_id,
        "model": str(llm),
        "pass": bool(passed),
        "failure_mode": failure_mode,
        "final_answer": final_answer,
        "normalized_answer": normalized_answer,
        "raw_output": response,
        "parsed_output": parsed,
        "prompt": prompt,
        "tokens_input": safe_get_attr(llm, "last_input_tokens"),
        "tokens_output": safe_get_attr(llm, "last_output_tokens"),
        "cost": safe_get_attr(llm, "last_cost"),
        "latency_ms": safe_get_attr(llm, "last_latency_ms"),
    }

def normalize_numeric_answer(text):
    if text is None:
        return ""

    s = str(text).strip().lower()
    replacements = {
        "$": "",
        ",": " ",
        "rad": "",
        "radian": "",
        "radians": "",
    }
    for old, new in replacements.items():
        s = s.replace(old, new)

    s = re.sub(r"\s+", " ", s).strip()
    return s

def extract_first_number(text):
    s = normalize_numeric_answer(text)
    if not s:
        return None, s

    m = re.search(r"[-+]?\d+(?:\.\d+)?", s)
    if not m:
        return None, s

    try:
        return float(m.group(0)), s
    except Exception:
        return None, s

def code_verifier(answer_text):
    value, normalized = extract_first_number(answer_text)

    if value is None:
        return False, "hallucination", normalized

    if abs(value - CANONICAL_NUMERIC_ANSWER) <= ABS_TOL:
        return True, None, normalized

    return False, None, normalized

def classify_failure_fp_0006(answer_text):
    value, normalized = extract_first_number(answer_text)

    if value is None:
        return "hallucination"

    # Common wrong-answer clusters from uploaded results
    if abs(value - 0.430) <= 0.01 or abs(value - 0.450) <= 0.01 or abs(value - 0.543) <= 0.02:
        return "misapplication_of_equation_or_model"

    if abs(value - 0.977) <= 0.02:
        return "hallucination"

    if abs(value - 2.36) <= 0.05 or abs(value - 2.62) <= 0.05 or abs(value - 2.76) <= 0.05:
        return "failure_to_recognize_key_aspects"

    if abs(value - CANONICAL_NUMERIC_ANSWER) <= 0.03:
        return "calculation_error"

    return "misapplication_of_equation_or_model"

# ----------------------------
# Frontier Physics Task 006
# ----------------------------
@kbench.task(
    name="FP-0006 Rotating Triatomic Molecule Phase Lag",
    description="Hard classical-mechanics / molecular-vibrations task on an Eckart-projected gyroscopic quadratic eigenproblem in a rotating frame."
)
def fp_0006_rotating_triatomic_phase_lag(llm) -> tuple[int, int]:
    prompt = r"""You are solving a physics problem. Return valid JSON only — no prose outside the JSON.

Work in reduced units where the bond-length unit $L=1$, the atomic mass unit $u=1$, and the stiffness scale $k_0=1$. Thus masses are pure numbers, all lengths are dimensionless, and time is measured in units $\sqrt{u/k_0}$. In these units, $\Omega$ is dimensionless.

A planar triatomic molecule $A_1\text{–}B\text{–}A_2$ has equilibrium positions
$A_1=(-1,0)$, $B=(0,0)$, $A_2=(\frac{8}{5},-\frac{6}{5})$.
Masses are $m_1=19,\ m_2=12,\ m_3=21$.
The frame rotates at constant angular speed $\Omega=2$ about $+\hat z$.

Let the in-plane displacement vector be
$d=(d_{1x},d_{1y},d_{2x},d_{2y},d_{3x},d_{3y})$.

Eckart constraints:
$19d_{1x} + 12d_{2x} + 21d_{3x} = 0$
$19d_{1y} + 12d_{2y} + 21d_{3y} = 0$
$-19 d_{1y} + 21(\frac{6}{5} d_{3x} + \frac{8}{5}d_{3y}) = 0$

Define the equilibrium bond-unit vectors
$\hat u_1 = (-1,0)$,
$\hat u_2 = (\frac45, -\frac35)$,
and in-plane perpendiculars
$\hat t_1 = (0,-1)$,
$\hat t_2 = (\frac35,\frac45)$.

Define the three linear fields
$\varepsilon_1 = \hat u_1 \cdot (d_1-d_2)$
$\varepsilon_2 = \hat u_2 \cdot (d_3-d_2)$
$\beta = \hat t_1 \cdot (d_1-d_2) + \hat t_2 \cdot (d_3-d_2)$

The quadratic effective potential is
$V^{(2)} = \frac12 (6)\varepsilon_1^2 + \frac12 (5)\varepsilon_2^2 + (2)\varepsilon_1\varepsilon_2 + \frac12(1)\beta^2$.

The rotating-frame linearized dynamics are
$m_i \ddot d_i + 2m_i \Omega (\hat z \times \dot d_i) + \nabla_{d_i} V^{(2)} = 0$,
with motion restricted to the Eckart subspace.

For the lowest-frequency in-plane normal mode, using the harmonic convention
$d(t)=\Re(Qe^{+i\omega t})$ with $\omega>0$,
calculate the phase lag
$\Delta \equiv \arg\left(\frac{X_{2x}}{X_{1x}}\right) \pmod{2\pi}$.

Question: What is the value of $\Delta$ in radians, to 3 significant figures?

Return JSON only in the following format:
{
  "final_answer": "<numeric value in radians>"
}"""

    response = llm.prompt(prompt)
    parsed = extract_json(response)

    total_checks = 1
    passed_checks = 0
    final_answer = ""
    normalized_answer = ""
    failure_mode = None

    if parsed is None:
        failure_mode = "hallucination"
    else:
        final_answer = parsed.get("final_answer", "")
        code_result, code_failure, normalized_answer = code_verifier(final_answer)

        if code_result is True:
            passed_checks = 1
        else:
            failure_mode = code_failure or classify_failure_fp_0006(final_answer)

    trace = build_trace(
        task_id="fp_0006",
        llm=llm,
        prompt=prompt,
        response=response,
        parsed=parsed,
        final_answer=final_answer,
        normalized_answer=normalized_answer,
        passed=(passed_checks == 1),
        failure_mode=failure_mode,
    )
    TRACE_LOG.append(trace)

    return (passed_checks, total_checks)

In [ ]:
fp_0006_rotating_triatomic_phase_lag.run(kbench.llm)

In [ ]:
results = fp_0006_rotating_triatomic_phase_lag.evaluate(llm=[kbench.llm])
results.as_dataframe()

In [ ]:
import pandas as pd

trace_df = pd.DataFrame(TRACE_LOG)
trace_df[trace_df["task_id"] == "fp_0006"]